# BCH111 — pre-ictal evoked state-space biomarker (reproducible)

**Channel BCH111SR · seizures ≥ 2026-09-13 · evoked window 2–50 ms, 1–500 Hz
band-pass (LP500), Z_ss-detrended.**

This notebook rebuilds — from committed code (`src/preictal_biomarker`) — the
pre-ictal state analysis whose scratch generators were lost. Run it top to
bottom; every figure and statistical null is reproduced by hand. Heavy steps
(the feature build, the animations) are cached / flagged.

**Method, in one paragraph.** Each stimulus epoch's evoked response is reduced to
8 features (6 per-waveform: line length, RMS, variance, lag-1 autocorrelation,
peak-to-trough, max slope; 2 across-trial critical-slowing stats of the primary
amplitude: rolling variance and rolling AR1). Features are gain-corrected and
**Z_ss-detrended** (impedance drift regressed out). We standardize, take the
first **6 PCs** (~95 % var) and cluster into **k = 4 states** (k-means). We then
ask whether the brain sits in different states in the 30 min **before** seizures
than during a clean baseline (≥ 2 h from any seizure) — tested honestly against a
**circular-shift null** (below).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))        # repo root on path
for s in (sys.stdout, sys.stderr):
    try: s.reconfigure(encoding='utf-8')
    except Exception: pass
import numpy as np, pandas as pd
from IPython.display import Image, display
from src.db.store import Store
from src.dashboard.data_helpers import load_config
from src.preictal_biomarker import (config as C, features as F, states as S,
                                     occupancy as O, trajectory as TR,
                                     timeline as TL, figures as G, anim as A)
cfg = load_config(); store = Store(cfg['database']['path'])
evoked_dir = cfg['chronic_evoked']['evoked_output_dir']; db = cfg['database']['path']
print('analysis window:', C.ANALYSIS_START.date(), '->', C.ANALYSIS_END.date())
print('features:', C.FEATURES8)

## 1 · Per-epoch feature matrix

`build_feature_matrix` warms the LP500 2–50 ms sidecars (≈30 s/file, **one-time**;
cached to parquet), builds the peri-ictal matrix, attaches Z_ss by recording,
detrends the waveform features, and adds the two CSD features.
Pass `force=True` to rebuild from raw.

In [ ]:
df = F.build_feature_matrix(store, evoked_dir, db, force=False)
print(df.shape)
display(df[['t_epoch','phase','time_to_onset_sec']+C.FEATURES8].describe().T)

## 2 · PCA (6 PCs) + k-means (k = 4) states

In [ ]:
model = S.fit_states(df)
ev = model.explained_var
print('PC variance %:', [round(100*x,1) for x in ev], ' cum=', round(100*ev.sum(),1))
print('state counts:', model.df['state'].value_counts().sort_index().to_dict())
print('state character:', model.labels)
display(model.profiles.round(2))

## 3 · Anchor figure — states + profiles + occupancy
Compare this to the original `05b`/`05d` to confirm the re-derived states line up.

In [ ]:
pre  = S.preictal_mask(model.df)
base = S.baseline_mask(model.df)
occ  = O.observed(model.df, pre, base)
p = G.anchor_figure(model, occ, pre, os.path.join(C.OUT_DIR,'rebuild_00_anchor_states.png'))
display(Image(p))

## 4 · Occupancy vs the circular-shift null

**What the null is.** A "what if seizure timing had nothing to do with state?"
reference. We keep the observed state time-series fixed and rigidly rotate *all
seizure onset times* by one random offset, wrapping past the end of the record
back to the start (**circular shift**). We re-collect the 30-min pre-ictal windows
at these fake onsets and recompute occupancy — repeated `N` times. The shift
preserves the state autocorrelation **and** the inter-seizure spacing; it destroys
only the specific seizure→state alignment. p = fraction of surrogates at least as
far from the null median as the observed bar. With n = 7 seizures the null band is
wide, so modest "shifts" are within chance.

In [ ]:
lead = F.lead_onsets(store)
null = O.circular_shift_null(model.df, lead, n_surr=2000, seed=0)
for j in range(C.K_STATES):
    print(f"state {j}: observed={null['observed'][j]:.1f}%  null med={null['median'][j]:.1f}%"
          f"  95%=[{null['lo'][j]:.1f},{null['hi'][j]:.1f}]  p={null['p'][j]:.2f}")

In [ ]:
display(Image(G.occupancy_null_compare(null, os.path.join(C.OUT_DIR,'rebuild_05_occupancy_null_compare.png'), baseline=occ['baseline'])))
display(Image(G.null_worked_example(null, os.path.join(C.OUT_DIR,'rebuild_05_null_worked_example.png'))))
display(Image(G.null_schematic(model.df, lead, os.path.join(C.OUT_DIR,'rebuild_04_null_schematic.png'))))

## 5 · Per-seizure pre-ictal state trajectory (lead vs all)

In [ ]:
allon = F.scoped_onsets(store)
tj_lead = TR.per_seizure_trajectory(model.df, lead)
tj_all  = TR.per_seizure_trajectory(model.df, allon)
display(Image(G.per_seizure_trajectory_fig(tj_lead, os.path.join(C.OUT_DIR,'rebuild_06_traj_lead.png'), title_suffix=' — lead seizures')))
display(Image(G.per_seizure_trajectory_fig(tj_all,  os.path.join(C.OUT_DIR,'rebuild_06_traj_all.png'),  title_suffix=' — all seizures')))

## 6 · Continuous state timeline (1 week / 2 days / 24 h)

In [ ]:
display(Image(TL.render_timeline(model.df, lead, os.path.join(C.OUT_DIR,'rebuild_07_state_timeline.png'))))

## 7 · Pre-ictal trajectory animations (shared PCA space) — slow

A time cursor travels toward onset; each epoch lights up in PCA space and earlier
ones dim (comet trail). Two builds: **lead** seizures and **all** seizures. Saves
GIF (+ MP4 if ffmpeg). This cell is slow (hundreds of frames).

In [ ]:
r_lead = A.animate_set(model, lead, A.seizure_labels(lead), os.path.join(C.OUT_DIR,'anim','preictal_traj_lead'), progress=lambda i,n: None)
print('lead:', r_lead)
r_all  = A.animate_set(model, allon, A.seizure_labels(allon), os.path.join(C.OUT_DIR,'anim','preictal_traj_all'), step_min=3.0, progress=lambda i,n: None)
print('all:', r_all)
display(Image(r_lead['gif']))

## Caveats (bake into every panel)
- **Effective n = 7 lead seizures.** Minute-level p-values are pseudoreplicated;
  trust the seizure-level and circular-shift-null numbers.
- One channel (BCH111SR), one window (2–50 ms, LP500), one date range (Sep 13–22).
- States are **re-derived** from committed code; integer labels may differ from the
  original scratch figures, but their character (energy / high-AR1 / post-ictal) is
  annotated in `model.labels`.
